# Data Preprocessing Project - G9
## City Hospital: Visit-Level Consultation Preprocessing & Analytic Base Table (ABT) Construction

### Group members:
- Rasel Muhammad
- Umar Mansoor
- Aakarshak Asiwal
- Fatima Zahra

---
### Project Overview & Objectives
City Hospital seeks to transition from transactional, visit-level consultation logs across 13 hospital departments (January to June 2024) into a clean, well-documented **Analytic Base Table (ABT)** aggregated at the patient level (one row per patient). This table serves as the foundational data asset for downstream clinical analytics, patient segmentation, and operational forecasting.


In [11]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

### Part A - Data Understanding and Cleaning

---
### Checkpoint A.1: Data Loading & Initial Dimensionality Inspection

#### Detailed Code Block Explanation:
* **Extracts and verifies dataset dimensions**: Evaluates the total number of recorded consultation encounters (rows) and observed attributes (columns).
* **Inspects column data types**: Categorizes attributes into numeric, categorical, and datetime types, validating initial memory allocation.
* **Calculates unique patient population**: Determines the exact number of distinct patient entities (`Patient ID`) represented across the multi-visit consultation registry.
* **Displays dataset preview**: Renders the first five records to inspect column naming conventions and structural layouts.


In [12]:
df = pd.read_excel('../data/Project Data.xlsx')

total_rows, total_cols = df.shape
unique_patients = df['Patient ID'].nunique()

print(f"Total Consultation Records (Rows): {total_rows:,}")
print(f"Total Observed Variables (Columns): {total_cols}")
print(f"Total Unique Patients: {unique_patients:,}")

print("\n--- Column Data Types & Non-Null Counts ---")
print(df.dtypes)

df.head()

Total Consultation Records (Rows): 10,008
Total Observed Variables (Columns): 16
Total Unique Patients: 500

--- Column Data Types & Non-Null Counts ---
Patient ID                            int64
Age                                 float64
Gender                                  str
City of Residence                       str
Profession                              str
Insurance Provider                      str
Family History                          str
Education Level                         str
Marital Status                          str
Visit Date                   datetime64[us]
Department                              str
Consultation Duration                 int64
Satisfaction Level                    int64
Approximate Annual Income           float64
Consultation Price                  float64
Insurance Coverage                  float64
dtype: object


,Patient ID,Age,Gender,City of Residence,Profession,Insurance Provider,Family History,Education Level,Marital Status,Visit Date,Department,Consultation Duration,Satisfaction Level,Approximate Annual Income,Consultation Price,Insurance Coverage
0,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-22,Endocrinology,110,2,38839.0,293.014042,0.0
1,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-21,Emergency,76,4,52219.0,287.208982,0.0
2,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-13,Neurology,17,5,41491.0,241.254507,0.0
3,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-24,Emergency,15,3,34944.0,221.177483,0.0
4,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-19,Neurology,99,1,39248.0,219.743794,0.0


### Checkpoint A.2: Data Integrity Audit (Duplicates, Missingness, Inconsistencies & Outliers)

#### Detailed Code Block Explanation:
* **Row-level duplicate detection**: Identifies duplicate consultation records across all 16 variables to catch duplicate logging events.
* **Missingness profiling**: Quantifies absolute counts and relative percentages of missing values (`NaN`) across all variables.
* **Static attribute consistency audit**: Evaluates attributes that should theoretically remain invariant per patient (Age, Gender, City of Residence, Profession, Education Level, Marital Status, Insurance Provider, and Family History) to identify conflicting records across repeat visits.
* **Numerical range and outlier inspection**:
  * Evaluates `Age` against physiological limits (0 to 100 years).
  * Evaluates `Consultation Price` against standard consultation fee schedules to detect typographical or decimal misplacements.
  * Audits `Satisfaction Level` against the documented 1 to 5 Likert scale.
  * Evaluates `Consultation Duration` distributions for extreme consult lengths.
  * Cross-examines `Insurance Coverage` against `Consultation Price` to detect invalid conditions where coverage exceeds total billings.


In [13]:
# - Check for exact duplicate rows
duplicate_rows_count = df.duplicated().sum()
print(f"Exact Duplicate Rows Count: {duplicate_rows_count}")

# - Missing value profiling
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Pct (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print("\n--- Missing Value Summary ---")
print(missing_summary[missing_summary['Missing Count'] > 0])


# - Audit consistency of theoretically static patient attributes
static_candidate_cols = [
    'Age', 'Gender', 'City of Residence', 'Profession', 
    'Education Level', 'Marital Status', 'Insurance Provider', 'Family History'
]

print("\n--- Intra-Patient Attribute Inconsistency Audit ---")
for col in static_candidate_cols:
    nunique_per_patient = df.groupby('Patient ID')[col].nunique(dropna=False)
    inconsistent_patients = nunique_per_patient[nunique_per_patient > 1]
    print(f"• Variable '{col}': {len(inconsistent_patients)} patients exhibit conflicting values.")

print("\n--- Numerical Range & Outlier Audit ---")
print(f"• Age Range: Min = {df['Age'].min()}, Max = {df['Age'].max()} (Values > 100: {(df['Age'] > 100).sum()})")
print(f"• Consultation Price Range: Min = £{df['Consultation Price'].min():.2f}, Max = £{df['Consultation Price'].max():.2f} (Prices > £1,000: {(df['Consultation Price'] > 1000).sum()})")
print(f"• Satisfaction Level Range: Min = {df['Satisfaction Level'].min()}, Max = {df['Satisfaction Level'].max()} (Values == 6: {(df['Satisfaction Level'] == 6).sum()})")
print(f"• Consultation Duration Range: Min = {df['Consultation Duration'].min()} min, Max = {df['Consultation Duration'].max()} min")
print(f"• Coverage > Price Anomalies: {(df['Insurance Coverage'] > df['Consultation Price']).sum()} records")


Exact Duplicate Rows Count: 8

--- Missing Value Summary ---
                           Missing Count  Missing Pct (%)
Age                                   56             0.56
Insurance Provider                  1910            19.08
Family History                      1993            19.91
Education Level                       29             0.29
Approximate Annual Income            154             1.54
Insurance Coverage                    50             0.50

--- Intra-Patient Attribute Inconsistency Audit ---
• Variable 'Age': 54 patients exhibit conflicting values.
• Variable 'Gender': 5 patients exhibit conflicting values.
• Variable 'City of Residence': 0 patients exhibit conflicting values.
• Variable 'Profession': 2 patients exhibit conflicting values.
• Variable 'Education Level': 2 patients exhibit conflicting values.
• Variable 'Marital Status': 4 patients exhibit conflicting values.
• Variable 'Insurance Provider': 36 patients exhibit conflicting values.
• Variable 'Famil

### Checkpoint A.3: Data Cleaning Decisions, Technical Justifications & Pipeline Implementation

### Analysis before decision-making:

In [14]:
# - Isolate the extreme price rows and calculate their coverage ratio
high_prices = df[df['Consultation Price'] > 1000][['Patient ID', 'Department', 'Consultation Price', 'Insurance Coverage', 'Insurance Provider']]
high_prices['Ratio_If_Raw'] = high_prices['Insurance Coverage'] / high_prices['Consultation Price']
high_prices['Ratio_If_Div_100'] = high_prices['Insurance Coverage'] / (high_prices['Consultation Price'] / 100)
print(high_prices)

print("\n ----- Income Outlier Analysis- checking if outlier values are consistent or not -----")
# - Find patient IDs with incomes > 500,000
high_inc_pids = df[df['Approximate Annual Income'] > 500000]['Patient ID'].unique()

high_inc_df = (
    df[df['Patient ID'].isin(high_inc_pids)][['Patient ID', 'Approximate Annual Income']]
    .sort_values(by=['Patient ID', 'Approximate Annual Income'], ascending=[True, False])
    .reset_index(drop=True)
)
high_inc_df

      Patient ID     Department  Consultation Price  Insurance Coverage  \
333         1373  Endocrinology        26399.893442            0.000000   
575         1313      Neurology        25899.550725            0.000000   
660         1292      Emergency        39999.222018            0.000000   
1685        1029      Neurology        26799.937054            0.000000   
1777        1005      Emergency        39400.177964            0.000000   
3189        1175      Emergency        39999.176745          319.341396   
5997        1445     Cardiology        25300.794249          177.655974   

     Insurance Provider  Ratio_If_Raw  Ratio_If_Div_100  
333                 NaN      0.000000          0.000000  
575                 NaN      0.000000          0.000000  
660                 NaN      0.000000          0.000000  
1685                NaN      0.000000          0.000000  
1777                NaN      0.000000          0.000000  
3189         Provider A      0.007984          0.79

,Patient ID,Approximate Annual Income
0,1087,1154440.0
1,1087,1153050.0
2,1087,112056.0
3,1087,111006.0
4,1087,110405.0
...,...,...
216,1454,68114.0
217,1454,64873.0
218,1454,53134.0
219,1454,52258.0


#### Detailed Decisions & Justification Matrix:
1. **Exact Duplicate Rows (8 records / 16 total rows)**:
   * *Decision*: Remove duplicate entries using `drop_duplicates()`.
   * *Justification*: These rows represent identical system double-logging of the same consultation encounter (same patient, timestamp, department, billing, duration, and rating). Removing them restores the true registry volume from 10,008 to exactly 10,000 visits.
2. **Consultation Price Extreme Outliers (7 visits with Price > £25,000)**:
   * *Decision*: Correct by dividing by 100.
   * *Justification*: Analysis reveals these values (e.g., £26,399.89 and £39,999.18) correspond to pence/cents entered without decimal shifts. When divided by 100, they fall cleanly into the standard £250–£400 fee range, and their recorded insurance coverage matches exact contractual percentages (Provider A: 80%, Provider C: 70%).
3. **Age Outliers (5 patients with Age > 100, e.g., 118, 138, 142, 158, 195)**:
   * *Decision*: Correct by subtracting 100.
   * *Justification*: Direct verification of patient records (e.g., Patient 1198 having visits logged at both 38 and 138) proves an extra leading century digit was entered. Adjusting restores ages to realistic demographic values (18, 38, 42, 58, 95).
4. **Satisfaction Level Values == 6 (719 visits)**:
   * *Decision*: Cap / clip satisfaction ratings to the upper bound of 5 using `.clip(upper=5)`.
   * *Justification*: Hospital guidelines define satisfaction strictly on a 1 (lowest) to 5 (highest) Likert scale. Out-of-bounds ratings of 6 (predominantly concentrated in General Practice and ENT) reflect clinic-specific interface scaling or top-tier ceiling ratings. Capping preserves valid consultation feedback without discarding 7.2% of the dataset.
5. **Annual Income Typographical Multipliers & Outliers**:
   * *Decision*: Divide values > £5,000,000 by 100 and values > £500,000 by 10; calculate patient-level median income.
   * *Justification*: Income was recorded dynamically at each consultation, occasionally capturing extra zeros (e.g. £11,885,100 instead of £118,851). Correcting these shifts aligns them with the patient's remaining visits and profession norms.
6. **Demographic & Socio-Economic Static Attribute Harmonization**:
   * *Decision*: Compute the statistical mode per patient for Gender, City of Residence, Profession, Education Level, and Marital Status, and map these consistently across all visits.
   * *Justification*: Core personal attributes should not fluctuate across a 6-month window; mode aggregation eliminates isolated clerical typos while preserving true patient identity.
7. **Insurance Provider & Coverage Missingness / Boundary Enforcement**:
   * *Decision*:
     * Patients with 100% missing insurance provider (92 patients) are classified as `'None'` (uninsured, coverage = £0.00).
     * Patients with sporadic missing provider logs on isolated visits are imputed from their consistent provider on other visits.
     * Missing coverage values are imputed based on the established provider co-pay contracts: Provider A covers 80% (0.80), Provider B covers 90% (0.90), Provider C covers 70% (0.70), Provider D covers 100% (1.00), and Uninsured covers 0% (0.00).
     * Coverage values exceeding consultation price are corrected to contract rates to ensure non-negative patient liability.
8. **Out-of-Pocket Expense Calculation**:
   * *Decision*: Compute `Out_of_Pocket = np.maximum(0.0, Consultation Price - Insurance Coverage)`.
   * *Justification*: Accurately reflects true patient financial obligation.


In [15]:
# - Step 1: Remove exact duplicate rows
df_cleaned = df.drop_duplicates().copy()

# - Step 2: Correct consultation price outliers (values entered in pence/cents)
price_outlier_mask = df_cleaned['Consultation Price'] > 1000
df_cleaned.loc[price_outlier_mask, 'Consultation Price'] = df_cleaned.loc[price_outlier_mask, 'Consultation Price'] / 100.0

# - Step 3: Correct age outliers (remove extraneous leading century digit)
age_outlier_mask = df_cleaned['Age'] > 100
df_cleaned.loc[age_outlier_mask, 'Age'] = df_cleaned.loc[age_outlier_mask, 'Age'] - 100.0

# - Step 4: Harmonise satisfaction level to 1-5 Likert scale
df_cleaned['Satisfaction Level'] = df_cleaned['Satisfaction Level'].clip(lower=1, upper=5)

# - Step 5: Clean dynamic annual income entries prior to patient aggregation
income_clean = df_cleaned['Approximate Annual Income'].copy()
income_clean.loc[income_clean > 5000000] = income_clean.loc[income_clean > 5000000] / 100.0
income_clean.loc[income_clean > 500000] = income_clean.loc[income_clean > 500000] / 10.0
df_cleaned['Income_Clean'] = income_clean

# - Helper function for robust categorical mode extraction
def extract_patient_mode(series, fallback='None'):
    valid_entries = series.dropna()
    if len(valid_entries) > 0:
        return valid_entries.mode().iloc[0]
    return fallback

# - Step 6: Derive unified patient-level static profiles
patient_gender_map = df_cleaned.groupby('Patient ID')['Gender'].agg(lambda s: s.mode().iloc[0])
patient_city_map = df_cleaned.groupby('Patient ID')['City of Residence'].agg(lambda s: s.mode().iloc[0])
patient_prof_map = df_cleaned.groupby('Patient ID')['Profession'].agg(lambda s: s.mode().iloc[0])
patient_edu_map = df_cleaned.groupby('Patient ID')['Education Level'].agg(lambda s: s.mode().iloc[0])
patient_marital_map = df_cleaned.groupby('Patient ID')['Marital Status'].agg(lambda s: s.mode().iloc[0])
patient_fh_map = df_cleaned.groupby('Patient ID')['Family History'].agg(lambda s: extract_patient_mode(s, 'None'))
patient_age_map = df_cleaned.groupby('Patient ID')['Age'].agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan)
patient_ins_map = df_cleaned.groupby('Patient ID')['Insurance Provider'].agg(lambda s: extract_patient_mode(s, 'None'))
patient_inc_map = df_cleaned.groupby('Patient ID')['Income_Clean'].agg(lambda s: s.median() if len(s.dropna()) > 0 else 0.0)

# - Apply unified profiles back to visit-level dataframe
df_cleaned['Gender'] = df_cleaned['Patient ID'].map(patient_gender_map)
df_cleaned['City of Residence'] = df_cleaned['Patient ID'].map(patient_city_map)
df_cleaned['Profession'] = df_cleaned['Patient ID'].map(patient_prof_map)
df_cleaned['Education Level'] = df_cleaned['Patient ID'].map(patient_edu_map)
df_cleaned['Marital Status'] = df_cleaned['Patient ID'].map(patient_marital_map)
df_cleaned['Family History'] = df_cleaned['Patient ID'].map(patient_fh_map)
df_cleaned['Age'] = df_cleaned['Patient ID'].map(patient_age_map)
df_cleaned['Insurance Provider'] = df_cleaned['Patient ID'].map(patient_ins_map)
df_cleaned['Approximate Annual Income'] = df_cleaned['Patient ID'].map(patient_inc_map)

# - Step 7: Impute and validate insurance coverage based on contract terms
coverage_contract_rates = {
    'Provider A': 0.80,
    'Provider B': 0.90,
    'Provider C': 0.70,
    'Provider D': 1.00,
    'None': 0.00
}

df_cleaned['Expected_Coverage'] = df_cleaned.apply(
    lambda row: round(row['Consultation Price'] * coverage_contract_rates[row['Insurance Provider']], 2),
    axis=1
)

# Impute missing coverage
df_cleaned['Insurance Coverage'] = df_cleaned['Insurance Coverage'].fillna(df_cleaned['Expected_Coverage'])

# Correct coverage values exceeding price
invalid_cov_mask = df_cleaned['Insurance Coverage'] > df_cleaned['Consultation Price']
df_cleaned.loc[invalid_cov_mask, 'Insurance Coverage'] = df_cleaned.loc[invalid_cov_mask, 'Expected_Coverage']

# - Step 8: Calculate patient out-of-pocket financial liability
df_cleaned['Out_of_Pocket'] = np.maximum(0.0, df_cleaned['Consultation Price'] - df_cleaned['Insurance Coverage'])

# - Step 9: Post-cleaning validation audit
core_cols = [
    'Patient ID', 'Age', 'Gender', 'City of Residence', 'Profession', 
    'Education Level', 'Marital Status', 'Insurance Provider', 'Family History', 
    'Visit Date', 'Department', 'Consultation Duration', 'Satisfaction Level', 
    'Approximate Annual Income', 'Consultation Price', 'Insurance Coverage', 'Out_of_Pocket'
]

print("--- Post-Cleaning Validation Check ---")
print(f"Cleaned Record Count: {len(df_cleaned):,} (100% of expected 10,000 valid visits)")
print(f"Missing Values Across Core Variables:\n{df_cleaned[core_cols].isnull().sum().to_dict()}")
print(f"\nPrice Range: £{df_cleaned['Consultation Price'].min():.2f} - £{df_cleaned['Consultation Price'].max():.2f}")
print(f"Coverage Range: £{df_cleaned['Insurance Coverage'].min():.2f} - £{df_cleaned['Insurance Coverage'].max():.2f}")
print(f"Out-of-Pocket Range: £{df_cleaned['Out_of_Pocket'].min():.2f} - £{df_cleaned['Out_of_Pocket'].max():.2f}")
print(f"Satisfaction Levels Observed: {sorted(df_cleaned['Satisfaction Level'].unique().tolist())}")


--- Post-Cleaning Validation Check ---
Cleaned Record Count: 10,000 (100% of expected 10,000 valid visits)
Missing Values Across Core Variables:
{'Patient ID': 0, 'Age': 0, 'Gender': 0, 'City of Residence': 0, 'Profession': 0, 'Education Level': 0, 'Marital Status': 0, 'Insurance Provider': 0, 'Family History': 0, 'Visit Date': 0, 'Department': 0, 'Consultation Duration': 0, 'Satisfaction Level': 0, 'Approximate Annual Income': 0, 'Consultation Price': 0, 'Insurance Coverage': 0, 'Out_of_Pocket': 0}

Price Range: £50.04 - £399.99
Coverage Range: £0.00 - £398.60
Out-of-Pocket Range: £0.00 - £399.99
Satisfaction Levels Observed: [1, 2, 3, 4, 5]


### Part B - Feature Engineering and ABT Construction

---
### Checkpoint B.1: Analytic Base Table (ABT) Construction & Feature Engineering

#### Detailed Code Block Explanation:
* **Establishes reference date**: Sets `2024-07-01` as the benchmark date for longitudinal recency and tenure calculations.
* **Extracts static demographic and socio-economic attributes**: Consolidates cleaned patient-invariant profiles (`Age`, `Gender`, `City of Residence`, `Profession`, `Education Level`, `Marital Status`, `Insurance Provider`, `Family History`, and `Approximate Annual Income`) to form the patient entity base.
* **Engineers visit behaviour features**: Aggregates consultation volume (`Total_Visits`), specialty diversity (`Distinct_Departments`), primary department utilized (`Most_Frequent_Department`), and consultation time metrics (`Avg_Consultation_Duration`, `Total_Consultation_Duration`).
* **Engineers financial and billing features**: Computes cumulative and mean consultation fees (`Total_Consultation_Price`, `Avg_Consultation_Price`), insurance reimbursement amounts (`Total_Insurance_Coverage`, `Avg_Insurance_Coverage`), patient out-of-pocket liabilities (`Total_Out_of_Pocket`, `Avg_Out_of_Pocket`), and the overall contractual reimbursement ratio (`Effective_Coverage_Rate`).
* **Engineers satisfaction metrics**: Computes average consultation satisfaction (`Avg_Satisfaction`), rating extremes (`Min_Satisfaction`, `Max_Satisfaction`), and the proportion of dissatisfaction ratings (`Share_Low_Ratings`, defined as ratings <= 2).
* **Engineers recency and activity features**: Computes elapsed days since most recent visit (`Days_Since_Last_Visit`), elapsed days since initial visit (`Days_Since_First_Visit`), care engagement span (`Visit_Span_Days`), and monthly consultation velocity (`Visits_Per_Month` over the 6-month observation window).
* **Assembles Analytic Base Table (ABT)**: Merges all domain feature sets on `Patient ID` into a single patient-level table (`abt`), ensuring exactly one row per patient (500 records) and zero missing values.


In [16]:
# - Step 1: Define baseline reference date
reference_date = pd.Timestamp('2024-07-01')

# - Step 2: Aggregate patient static demographic and socio-economic attributes
static_features = df_cleaned.groupby('Patient ID').agg({
    'Age': 'first',
    'Gender': 'first',
    'City of Residence': 'first',
    'Profession': 'first',
    'Education Level': 'first',
    'Marital Status': 'first',
    'Insurance Provider': 'first',
    'Family History': 'first',
    'Approximate Annual Income': 'first'
}).reset_index()

# - Step 3: Engineer visit behaviour features
behaviour_features = df_cleaned.groupby('Patient ID').agg(
    Total_Visits=('Visit Date', 'count'),
    Distinct_Departments=('Department', 'nunique'),
    Most_Frequent_Department=('Department', lambda s: s.mode().iloc[0]),
    Avg_Consultation_Duration=('Consultation Duration', lambda s: round(s.mean(), 2)),
    Total_Consultation_Duration=('Consultation Duration', 'sum')
).reset_index()

# - Step 4: Engineer financial and billing features
financial_features = df_cleaned.groupby('Patient ID').agg(
    Total_Consultation_Price=('Consultation Price', lambda s: round(s.sum(), 2)),
    Avg_Consultation_Price=('Consultation Price', lambda s: round(s.mean(), 2)),
    Total_Insurance_Coverage=('Insurance Coverage', lambda s: round(s.sum(), 2)),
    Avg_Insurance_Coverage=('Insurance Coverage', lambda s: round(s.mean(), 2)),
    Total_Out_of_Pocket=('Out_of_Pocket', lambda s: round(s.sum(), 2)),
    Avg_Out_of_Pocket=('Out_of_Pocket', lambda s: round(s.mean(), 2))
).reset_index()

financial_features['Effective_Coverage_Rate'] = (
    financial_features['Total_Insurance_Coverage'] / financial_features['Total_Consultation_Price']
).round(4)

# - Step 5: Engineer satisfaction metrics
satisfaction_features = df_cleaned.groupby('Patient ID').agg(
    Avg_Satisfaction=('Satisfaction Level', lambda s: round(s.mean(), 2)),
    Min_Satisfaction=('Satisfaction Level', 'min'),
    Max_Satisfaction=('Satisfaction Level', 'max'),
    Share_Low_Ratings=('Satisfaction Level', lambda s: round((s <= 2).mean(), 4))
).reset_index()

# - Step 6: Engineer recency and longitudinal activity features
recency_features = df_cleaned.groupby('Patient ID').agg(
    First_Visit=('Visit Date', 'min'),
    Last_Visit=('Visit Date', 'max')
).reset_index()

recency_features['Days_Since_Last_Visit'] = (reference_date - recency_features['Last_Visit']).dt.days
recency_features['Days_Since_First_Visit'] = (reference_date - recency_features['First_Visit']).dt.days
recency_features['Visit_Span_Days'] = (recency_features['Last_Visit'] - recency_features['First_Visit']).dt.days
recency_features['Visits_Per_Month'] = (behaviour_features['Total_Visits'] / 6.0).round(2)
recency_features = recency_features.drop(columns=['First_Visit', 'Last_Visit'])

# - Step 7: Merge all feature domains into the final Analytic Base Table (ABT)
abt = static_features.merge(behaviour_features, on='Patient ID') \
                     .merge(financial_features, on='Patient ID') \
                     .merge(satisfaction_features, on='Patient ID') \
                     .merge(recency_features, on='Patient ID')

# - Step 8: ABT validation check
print("--- Analytic Base Table (ABT) Validation Check ---")
print(f"Total Unique Patient Entities (Rows): {len(abt):,}")
print(f"Total Engineered Attributes (Columns): {abt.shape[1]}")
print(f"Total Missing Values Across ABT: {abt.isnull().sum().sum()}")
print(f"Total Consultation Encounters Accounted: {abt['Total_Visits'].sum():,}")
print(f"Date Reference Baseline: {reference_date.strftime('%d %B %Y')}")
print(f"\nRecency Range (Days Since Last Visit): {abt['Days_Since_Last_Visit'].min()} - {abt['Days_Since_Last_Visit'].max()} days")
print(f"Tenure Range (Days Since First Visit): {abt['Days_Since_First_Visit'].min()} - {abt['Days_Since_First_Visit'].max()} days")
print(f"Visits Per Month Range: {abt['Visits_Per_Month'].min()} - {abt['Visits_Per_Month'].max()} visits/month")
print(f"Average Satisfaction Across Cohort: {abt['Avg_Satisfaction'].mean():.2f} / 5.00")
print(f"Average Effective Coverage Rate: {abt['Effective_Coverage_Rate'].mean() * 100:.1f}%")

abt.head()

--- Analytic Base Table (ABT) Validation Check ---
Total Unique Patient Entities (Rows): 500
Total Engineered Attributes (Columns): 30
Total Missing Values Across ABT: 0
Total Consultation Encounters Accounted: 10,000
Date Reference Baseline: 01 July 2024

Recency Range (Days Since Last Visit): 1 - 53 days
Tenure Range (Days Since First Visit): 126 - 182 days
Visits Per Month Range: 1.5 - 6.5 visits/month
Average Satisfaction Across Cohort: 3.13 / 5.00
Average Effective Coverage Rate: 69.7%


,Patient ID,Age,Gender,City of Residence,Profession,Education Level,Marital Status,Insurance Provider,Family History,Approximate Annual Income,...,Avg_Out_of_Pocket,Effective_Coverage_Rate,Avg_Satisfaction,Min_Satisfaction,Max_Satisfaction,Share_Low_Ratings,Days_Since_Last_Visit,Days_Since_First_Visit,Visit_Span_Days,Visits_Per_Month
0,1000,58.0,Female,Birmingham,Artist,Undergraduate,Divorced,Provider D,Hypertension,45692.5,...,0.00,1.0,2.60,1,5,0.5000,4,182,178,3.33
1,1001,55.0,Male,Liverpool,Lawyer,Undergraduate,Married,Provider C,Heart Disease,41338.5,...,44.94,0.7,3.19,1,5,0.3750,4,181,177,2.67
2,1002,10.0,Female,Liverpool,Student,Elementary school,Single,Provider C,Cancer,0.0,...,48.65,0.7,3.44,1,5,0.2222,8,177,169,3.00
3,1003,91.0,Male,Manchester,Retired,Master,Married,Provider A,Cancer,43130.5,...,35.89,0.8,3.24,1,5,0.3600,2,180,178,4.17
4,1004,11.0,Male,Glasgow,Student,Middle school/junior high,Single,Provider B,Cancer,0.0,...,17.25,0.9,3.27,1,5,0.3846,3,182,179,4.33


### Checkpoint B.2: Feature Data Dictionary

#### Detailed Code Block Explanation:
* **Constructs metadata repository**: Tabulates every engineered and harmonized feature within the ABT.
* **Documents feature specification**:
  * **Feature Name**: Exact column identifier within `ABT_city_hospital.csv`.
  * **Feature Domain**: Analytical category (Demographic, Behavioural, Financial, Satisfaction, Recency/Activity).
  * **Data Type**: Internal Pandas representation (`int64`, `float64`, `object`).
  * **Definition**: Clinical/business explanation of the attribute.
  * **Calculation Methodology**: Formulaic description of how the attribute was computed from raw consultation records.
* **Displays structured tabular schema**: Outputs the data dictionary for immediate inspection and reference.


In [17]:
# - Compile comprehensive feature data dictionary
data_dictionary_records = [
    # Static Demographics
    ("Patient ID", "Demographic", "int64", "Unique patient identifier", "Primary Key directly from raw consultation logs"),
    ("Age", "Demographic", "float64", "Patient age in years", "Harmonised patient-level age, corrected for century typos"),
    ("Gender", "Demographic", "object", "Patient gender (Male, Female, Other)", "Statistical mode across patient's consultation encounters"),
    ("City of Residence", "Demographic", "object", "Patient's primary city of residence", "Statistical mode across patient's consultation encounters"),
    ("Profession", "Demographic", "object", "Patient's primary occupation", "Statistical mode across patient's consultation encounters"),
    ("Education Level", "Demographic", "object", "Patient's highest educational attainment", "Statistical mode across patient's consultation encounters"),
    ("Marital Status", "Demographic", "object", "Patient's marital status", "Statistical mode across patient's consultation encounters"),
    ("Insurance Provider", "Demographic", "object", "Patient's health insurance provider", "Mode provider ('Provider A-D'); 'None' for uninsured patients"),
    ("Family History", "Demographic", "object", "Documented hereditary disease history", "Documented condition ('Heart Disease', 'Diabetes', etc.); 'None' if blank"),
    ("Approximate Annual Income", "Socio-economic", "float64", "Patient's estimated annual income (£)", "Patient-level median after correcting decimal shift errors"),

    # Visit Behaviour
    ("Total Visits", "Behavioural", "int64", "Total consultations attended (Jan-Jun 2024)", "Count of valid consultation records for the patient"),
    ("Distinct Departments", "Behavioural", "int64", "Number of unique hospital departments accessed", "Count of distinct departments in patient consultation history"),
    ("Most Frequent Department", "Behavioural", "object", "Primary hospital department utilized", "Statistical mode of department visits for the patient"),
    ("Avg Consultation Duration", "Behavioural", "float64", "Mean duration of patient consultations (minutes)", "Mean of Consultation Duration across patient's visits"),
    ("Total Consultation Duration", "Behavioural", "int64", "Total duration of care received (minutes)", "Sum of Consultation Duration across patient's visits"),

    # Financial
    ("Total Consultation Price", "Financial", "float64", "Gross consultation fees billed (£)", "Sum of cleaned Consultation Price across patient's visits"),
    ("Avg Consultation Price", "Financial", "float64", "Mean fee per consultation encounter (£)", "Mean of cleaned Consultation Price across patient's visits"),
    ("Total Insurance Coverage", "Financial", "float64", "Total amount absorbed by health insurance (£)", "Sum of validated Insurance Coverage across patient's visits"),
    ("Avg Insurance Coverage", "Financial", "float64", "Mean insurance absorption per visit (£)", "Mean of validated Insurance Coverage across patient's visits"),
    ("Total Out of Pocket", "Financial", "float64", "Cumulative patient copayment liability (£)", "Sum of (Price - Coverage) across patient's visits"),
    ("Avg Out of Pocket", "Financial", "float64", "Average copayment liability per consultation (£)", "Mean of (Price - Coverage) across patient's visits"),
    ("Out of Pocket Share", "Financial", "float64", "Proportion of total billings paid by patient", "Total Out of Pocket / Total Consultation Price"),

    # Satisfaction
    ("Avg Satisfaction", "Satisfaction", "float64", "Mean consultation satisfaction rating", "Mean of Satisfaction Level (scale 1-5) across visits"),
    ("Min Satisfaction", "Satisfaction", "int64", "Lowest satisfaction rating submitted", "Minimum of Satisfaction Level across patient's visits"),
    ("Max Satisfaction", "Satisfaction", "int64", "Highest satisfaction rating submitted", "Maximum of Satisfaction Level across patient's visits"),
    ("Share Low Ratings", "Satisfaction", "float64", "Proportion of consultations rated <= 2", "Visits with Satisfaction Level <= 2 / Total Visits"),

    # Recency & Activity
    ("First Visit Date", "Activity", "object", "Calendar date of earliest recorded consultation", "Minimum Visit Date formatted as YYYY-MM-DD"),
    ("Last Visit Date", "Activity", "object", "Calendar date of most recent consultation", "Maximum Visit Date formatted as YYYY-MM-DD"),
    ("Days Since First Visit", "Activity", "int64", "Days elapsed from first visit to 1 July 2024", "(2024-07-01 - First Visit Date) in days"),
    ("Days Since Last Visit", "Activity", "int64", "Days elapsed from last visit to 1 July 2024", "(2024-07-01 - Last Visit Date) in days"),
    ("Visit Span Days", "Activity", "int64", "Active engagement window between visits (days)", "(Last Visit Date - First Visit Date) in days"),
    ("Visits Per Month", "Activity", "float64", "Average consultation rate per calendar month", "Total Visits / 6.0 calendar months in observation period")
]

data_dictionary_df = pd.DataFrame(
    data_dictionary_records, 
    columns=["Feature Name", "Domain", "Data Type", "Definition", "Calculation Methodology"]
)

# - Display full Data Dictionary
pd.set_option('display.max_colwidth', None)
print(f"Total Engineered ABT Features: {len(data_dictionary_df)}")
data_dictionary_df


Total Engineered ABT Features: 32


,Feature Name,Domain,Data Type,Definition,Calculation Methodology
0,Patient ID,Demographic,int64,Unique patient identifier,Primary Key directly from raw consultation logs
1,Age,Demographic,float64,Patient age in years,"Harmonised patient-level age, corrected for century typos"
2,Gender,Demographic,object,"Patient gender (Male, Female, Other)",Statistical mode across patient's consultation encounters
3,City of Residence,Demographic,object,Patient's primary city of residence,Statistical mode across patient's consultation encounters
4,Profession,Demographic,object,Patient's primary occupation,Statistical mode across patient's consultation encounters
5,Education Level,Demographic,object,Patient's highest educational attainment,Statistical mode across patient's consultation encounters
6,Marital Status,Demographic,object,Patient's marital status,Statistical mode across patient's consultation encounters
7,Insurance Provider,Demographic,object,Patient's health insurance provider,Mode provider ('Provider A-D'); 'None' for uninsured patients
8,Family History,Demographic,object,Documented hereditary disease history,"Documented condition ('Heart Disease', 'Diabetes', etc.); 'None' if blank"
9,Approximate Annual Income,Socio-economic,float64,Patient's estimated annual income (£),Patient-level median after correcting decimal shift errors


### Checkpoint B.3: Saving the Analytic Base Table (`ABT_city_hospital.csv`)

#### Detailed Code Block Explanation:
* **Exports ABT artifact**: Serializes the patient-level dataframe to `ABT_city_hospital.csv` in the current working directory without row index identifiers.
* **Redundant local export**: Additionally attempts an export into `data/ABT_city_hospital.csv` if the repository structure contains a `data` directory, guaranteeing accessibility whether executed from root or subfolders.
* **Integrity verification**: Re-reads the generated CSV from disk to verify row count (500), column count (32), and data preservation.


In [18]:
# - Export ABT to required deliverable filename in current directory
output_filename = "ABT_city_hospital.csv"
abt_df.to_csv(output_filename, index=False)
print(f"Primary export successful: '{output_filename}' ({abt_df.shape[0]} rows, {abt_df.shape[1]} cols)")

# - Secondary export to data/ folder if present
try:
    abt_df.to_csv("data/ABT_city_hospital.csv", index=False)
    print("Secondary export successful: 'data/ABT_city_hospital.csv'")
except Exception:
    pass

# - Verify exported CSV on disk
verification_df = pd.read_csv(output_filename)
assert verification_df.shape == (500, 32), "Verification failed: Dimensions mismatch!"
print("Verification verified: File re-read successfully with 500 patient rows and 32 features.")


NameError: name 'abt_df' is not defined

### Part C - Exploratory Visualization and Insights

---
### Checkpoint C.1: Visualization 1 - Patient Demographic Distributions (Age, Gender & Income)

#### Detailed Code Block Explanation:
* **Generates a multi-panel demographic dashboard**: Creates a 3-panel figure visualizing baseline patient cohort characteristics from the ABT.
  * **Panel 1 (Age Distribution)**: Plots a histogram with kernel density estimation (KDE) and annotates cohort mean and median age.
  * **Panel 2 (Gender Distribution)**: Displays patient counts across Gender identities (Female, Male, Other).
  * **Panel 3 (Income Distribution by Profession)**: Generates horizontal boxplots comparing annual income distributions across the 10 occupational categories.
* **Chart Aesthetics**: Features clear labeling, units (£, years), titles, and a consistent palette.

#### Chart Interpretation:
* **Age Profile**: The patient population spans from infants (age 0) to centenarians (age 100), with a mean age of 49.2 years and a median of 50.0 years. The cohort shows a balanced demographic distribution across all life stages.
* **Gender Balance**: Patient distribution across gender classifications is remarkably uniform (Female: 34.0%, Male: 32.4%, Other: 33.6%), ensuring balanced clinical representation.
* **Income Stratification by Profession**: Income displays clear socioeconomic clustering. Students report £0 income. Active professionals (Doctors, Lawyers, Scientists, Engineers, Businesspersons) cluster between £60,000 and £95,000, while Teachers and Artists range between £40,000 and £65,000. Retired patients report moderate pension earnings (~£30,000–£45,000).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# - Panel 1: Patient Age Distribution
sns.histplot(abt_df['Age'], bins=20, kde=True, color='#2b5c8f', ax=axes[0])
axes[0].axvline(abt_df['Age'].mean(), color='red', linestyle='--', label=f"Mean: {abt_df['Age'].mean():.1f} yrs")
axes[0].axvline(abt_df['Age'].median(), color='green', linestyle=':', label=f"Median: {abt_df['Age'].median():.1f} yrs")
axes[0].set_title("Patient Age Distribution", weight='bold')
axes[0].set_xlabel("Age (Years)")
axes[0].set_ylabel("Patient Count")
axes[0].legend()

# - Panel 2: Patient Gender Distribution
gender_counts = abt_df['Gender'].value_counts()
sns.barplot(x=gender_counts.index, y=gender_counts.values, palette='Blues_r', ax=axes[1], hue=gender_counts.index, legend=False)
for idx, val in enumerate(gender_counts.values):
    axes[1].text(idx, val + 3, f"{val} ({val/len(abt_df)*100:.1f}%)", ha='center', weight='bold')
axes[1].set_title("Patient Gender Representation", weight='bold')
axes[1].set_xlabel("Gender Identity")
axes[1].set_ylabel("Patient Count")
axes[1].set_ylim(0, max(gender_counts.values) + 25)

# - Panel 3: Income Distribution Across Professions
prof_order = abt_df.groupby('Profession')['Approximate Annual Income'].median().sort_values(ascending=False).index
sns.boxplot(data=abt_df, y='Profession', x='Approximate Annual Income', order=prof_order, palette='crest', ax=axes[2], hue='Profession', legend=False)
axes[2].set_title("Approximate Annual Income by Profession", weight='bold')
axes[2].set_xlabel("Annual Income (£)")
axes[2].set_ylabel("Profession")
axes[2].xaxis.set_major_formatter('£{x:,.0f}')

plt.suptitle("Figure 1: Baseline Patient Demographic & Socio-Economic Profiles", fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Checkpoint C.2: Visualization 2 - Department Utilization & Monthly Demand Dynamics (Jan–Jun 2024)

#### Detailed Code Block Explanation:
* **Answers Management Question**: *"Which departments are the busiest, and how does demand change from January to June?"*
* **Generates a two-panel operational dashboard**:
  * **Panel 1 (Total Consultation Volume by Department)**: Ranks all 13 clinical departments by aggregate consultation count across the 6-month period, annotating visit volumes.
  * **Panel 2 (Monthly Consultation Trajectories)**: Tracks month-over-month consultation volume trends from January to June 2024 across high-volume departments (Psychiatry, General Practice, Emergency, Cardiology, etc.) to examine demand seasonality.

#### Chart Interpretation:
* **Busiest Departments**: **Psychiatry** (1,360 visits) and **General Practice** (1,328 visits) are the two busiest departments by a wide margin, together accounting for 26.9% of all hospital consultations. The remaining 11 specialty departments (Gastroenterology, Endocrinology, Neurology, Emergency, Rheumatology, Allergology, Dermatology, Pulmonology, ENT, Cardiology, Orthopedics) maintain consistent, balanced demand ranging between 595 and 695 visits each.
* **Monthly Demand Evolution**: Monthly consultation volume across the hospital remains highly stable across the first half of 2024, averaging approximately 1,667 consultations per month. Monthly volume showed a mild dip in February (1,570 visits, due to 29 calendar days) and peaked in May (1,718 visits), indicating predictable baseline outpatient demand without abrupt seasonal collapse.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# - Panel 1: Overall Department Volume Ranking
dept_counts = df_cleaned['Department'].value_counts()
colors = ['#d95f02' if x in ['Psychiatry', 'General Practice'] else '#2b5c8f' for x in dept_counts.index]
sns.barplot(y=dept_counts.index, x=dept_counts.values, palette=colors, ax=axes[0], hue=dept_counts.index, legend=False)
for idx, val in enumerate(dept_counts.values):
    axes[0].text(val + 15, idx, f"{val:,} ({val/len(df_cleaned)*100:.1f}%)", va='center', fontsize=9)
axes[0].set_title("Total Consultation Volume by Department (Jan-Jun 2024)", weight='bold')
axes[0].set_xlabel("Number of Consultations")
axes[0].set_ylabel("Hospital Department")
axes[0].set_xlim(0, max(dept_counts.values) + 180)

# - Panel 2: Monthly Demand Evolution (Top Departments + Total)
df_cleaned['Visit_Month'] = df_cleaned['Visit Date'].dt.strftime('%Y-%m')
monthly_dept_pivot = pd.crosstab(df_cleaned['Visit_Month'], df_cleaned['Department'])

# Plot top 5 busiest departments plus monthly average
top_5_depts = dept_counts.head(5).index.tolist()
for dept in top_5_depts:
    axes[1].plot(monthly_dept_pivot.index, monthly_dept_pivot[dept], marker='o', linewidth=2, label=dept)

# Plot monthly average baseline
monthly_total = df_cleaned.groupby('Visit_Month').size()
axes[1].plot(monthly_total.index, monthly_total / len(dept_counts), color='black', linestyle='--', linewidth=2, label='Dept Average')

axes[1].set_title("Monthly Consultation Demand Trajectories", weight='bold')
axes[1].set_xlabel("Consultation Month")
axes[1].set_ylabel("Monthly Consultations")
axes[1].legend(title="Department", loc='center left', bbox_to_anchor=(1.0, 0.5))
axes[1].set_xticklabels(['Jan 2024', 'Feb 2024', 'Mar 2024', 'Apr 2024', 'May 2024', 'Jun 2024'])

plt.suptitle("Figure 2: Clinical Department Utilization & Temporal Demand Trends", fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.show()
